# Ariel 2025 — Notebook 3a: Build Sequence Tensors (CPU)

Chạy notebook này **một lần trên CPU** để precompute tensor chuỗi từ raw light curves rồi push lên branch `running`.

Notebook 3 (GPU) sẽ load tensor trực tiếp từ repo thay vì build lại từ đầu.

> **Không cần GPU.** Chạy xong commit tensor vào `precomputed/` và push lên GitHub.

In [ ]:
# === Setup: clone running branch ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src", "src", "../src"]:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using ariel_ml from:", _p); break
else:
    print("WARNING: ariel_ml source not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Cấu hình

Giữ nguyên các giá trị này khi chạy Notebook 3 (GPU) để load đúng file.

In [ ]:
import numpy as np
import pandas as pd

from config import PreprocessConfig, DatasetConfig
from data_io import ArielDataRepository
from pipeline import ArielPreprocessFeaturePipeline
from sequence_dataset import build_sequence_dataset
from training import make_train_validation_split

LIMIT = 300
TIME_STEPS = 128
WAVELENGTH_BINS = 64
RANDOM_STATE = 42

print(f"Config: LIMIT={LIMIT}, TIME_STEPS={TIME_STEPS}, WAVELENGTH_BINS={WAVELENGTH_BINS}")

## 2. Dựng tensor chuỗi từ raw light curves

Bước chậm nhất — chỉ cần chạy một lần.

In [ ]:
repository = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
pipeline = ArielPreprocessFeaturePipeline(
    PreprocessConfig(target_time_bins=TIME_STEPS, apply_cds=True, detrend_degree=2, smooth_window=5)
)

dataset = build_sequence_dataset(
    repository, pipeline, "train",
    limit=LIMIT, wavelength_bins=WAVELENGTH_BINS, on_error="skip",
)
print("Sequence tensor:", dataset.x.shape, "| planets kept:", len(dataset.planet_ids),
      "| failures:", len(dataset.failures))

## 3. Align với target, split, chuẩn hóa

In [ ]:
targets = pd.read_csv(DATA_ROOT / "train.csv")
targets["planet_id"] = targets["planet_id"].astype(str)
target_cols = [c for c in targets.columns if c != "planet_id"]
tmap = targets.set_index("planet_id")[target_cols]

mask = [pid in tmap.index for pid in dataset.planet_ids]
X = dataset.x[np.array(mask)]
ids = [pid for pid, keep in zip(dataset.planet_ids, mask) if keep]
Y = tmap.loc[ids].to_numpy(dtype=float)
print("X:", X.shape, "| Y:", Y.shape)

tr, va = make_train_validation_split(n_samples=X.shape[0], validation_fraction=0.2, random_state=RANDOM_STATE)
mean = X[tr].mean(axis=(0, 1), keepdims=True)
std  = X[tr].std(axis=(0, 1), keepdims=True) + 1e-8
Xn   = (X - mean) / std
print("train:", len(tr), "val:", len(va))

## 4. Lưu tensor vào `precomputed/`

In [ ]:
save_path = PRECOMPUTED_DIR / f"sequence_train_L{LIMIT}_T{TIME_STEPS}_W{WAVELENGTH_BINS}.npz"
np.savez_compressed(save_path, Xn=Xn, Y=Y, tr=tr, va=va, planet_ids=np.array(ids))
size_mb = save_path.stat().st_size / 1e6
print(f"Saved → {save_path}  ({size_mb:.1f} MB)")
print(f"  Xn: {Xn.shape}, Y: {Y.shape}, train: {len(tr)}, val: {len(va)}")